**OBSOLETE -- superseded by the 2026-08-25 reorientation plan.**

Produced the 0.686 gold-agreement baseline that A1a' (validating four
already-published LLM label sets) and A1a (the re-derived lexicon, if
needed) are measured against -- that number is already recorded in
README.md/memory, so this notebook itself doesn't need to stay active.
Kept as a historical record, not maintained going forward. See
README.md for the current plan.

# 03 - Validacion del report-labeler

Objetivo (plan Fase 3): construir label_report() en src/labelers.py y validarlo contra el subset gold antes de usarlo como target de entrenamiento. Gate: debe superar un baseline trivial (prevalencia global por hallazgo).

In [ ]:
# Corre en Kaggle o en local (contra data/raw/, export de la Fase 2) -
# mismo patron de auto-deteccion que 02_eda_reports.ipynb. No importa de
# src/ (Kaggle no tiene el repo montado, solo el notebook en si) - se
# repite aqui una copia de macro_roc_auc/per_finding_roc_auc, misma
# convencion que las constantes duplicadas en 01/02. Mantener
# sincronizado con src/evaluate.py si ese archivo cambia.
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score


def macro_roc_auc(y_true, y_pred):
    if list(y_true.columns) != list(y_pred.columns):
        raise ValueError("y_true and y_pred must have matching columns.")
    per_finding_auc = {
        col: roc_auc_score(y_true[col], y_pred[col])
        for col in y_true.columns
    }
    return float(np.mean(list(per_finding_auc.values())))


def per_finding_roc_auc(y_true, y_pred):
    return pd.Series(
        {col: roc_auc_score(y_true[col], y_pred[col]) for col in y_true.columns}
    )


_KAGGLE_RAW = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
_LOCAL_RAW = Path("../data/raw")
RAW_DIR = _KAGGLE_RAW if _KAGGLE_RAW.exists() else _LOCAL_RAW
assert RAW_DIR.exists(), f"Dataset no encontrado ni en {_KAGGLE_RAW} ni en {_LOCAL_RAW}."
print("Corriendo contra:", RAW_DIR)

OFFICIAL_LABEL_COLUMNS = {
    "acl_injury": "ACL",
    "mcl_injury": "MCL",
    "medial_meniscus_tear": "Medial Meniscus",
    "lateral_meniscus_tear": "Lateral Meniscus",
    "oa_medial_compartment": "Medial OA",
    "oa_lateral_compartment": "Lateral OA",
    "oa_patellofemoral_compartment": "PF OA",
    "effusion": "Effusion",
    "synovitis": "Synovitis",
    "bakers_cyst": "Baker's",
    "bone_contusion": "Contusion",
    "fracture": "Fracture",
}
FINDINGS = list(OFFICIAL_LABEL_COLUMNS.keys())
LABEL_COLS = list(OFFICIAL_LABEL_COLUMNS.values())

## A. Cargar el gold y confirmar el tipo real de las 12 etiquetas

El notebook de referencia prvsiyan describe las 12 columnas como "graded targets" (targets graduados, tipo severidad 0-3). No darlo por bueno sin medirlo en nuestros propios datos — el mismo principio que en Fase 1/2.

In [ ]:
train = pd.read_csv(RAW_DIR / "train.csv")
n_labels_present = train[LABEL_COLS].notna().sum(axis=1)
gold = train.loc[n_labels_present == len(LABEL_COLS)].copy()
print(f"gold: {len(gold)} filas")

print("\nValores unicos por columna (gold):")
for c in LABEL_COLS:
    print(f"  {c}: {sorted(gold[c].unique().tolist())}")

print("\nTasa base (fraccion positiva) por hallazgo:")
print(gold[LABEL_COLS].mean().sort_values())

**Resultado real:** las 12 columnas son binarias (0.0/1.0), no graduadas — contradice la descripcion del notebook de referencia. Esto simplifica la validacion: `label_report()` puede compararse directo contra el gold vía `macro_roc_auc` (ground truth binario, prediccion continua en [0,1]), sin necesitar umbralizar ningun grado. Tasa base minima: MCL con 9/58 (~15.5%) positivos — suficiente para que ROC-AUC este definida en las 12 columnas (`macro_roc_auc` lanzaria error si alguna tuviera una sola clase).

## B. Normalizar, deshacer hard-wrap, segmentar en clausulas

Misma logica que el notebook de referencia (normalizar -> unwrap -> segmentar), con el `unwrap()` corregido: en la Fase 2 (`02_eda_reports.ipynb` seccion C) se encontro que el heuristico original confundia un simbolo de vineta (`>`) con "no empieza en mayuscula", marcando lineas independientes como continuacion de la anterior. Aqui se retira el prefijo de vineta antes de comprobar la mayuscula (pero se conserva en el texto unido).

In [ ]:
def normalize(text):
    if not isinstance(text, str):
        return ""
    t = text.lower()
    t = unicodedata.normalize("NFKD", t)
    t = "".join(ch for ch in t if not unicodedata.combining(ch))
    t = re.sub(r"[_\-/\\]+", " ", t)
    t = re.sub(r"[ \t]+", " ", t)
    return t


_BULLET_PREFIX_RE = re.compile(r"^[>*\u2022\-]+\s*")


def unwrap(text):
    """Rejoin lines a fixed-width layout broke mid-sentence (bullet-aware)."""
    if not isinstance(text, str):
        return ""
    out = []
    for line in text.split("\n"):
        s = line.strip()
        s_check = _BULLET_PREFIX_RE.sub("", s)  # fix Fase 2: no confundir ">" con "no-mayuscula"
        if (out and out[-1] and not re.search(r"[.;:!?>*\u2022]$", out[-1])
                and len(out[-1].split()) >= 4 and s_check and not s_check[:1].isupper()):
            out[-1] = out[-1] + " " + s
        else:
            out.append(s)
    return "\n".join(out)


_SENT_SPLIT_RE = re.compile(r"(?<=[.;!?])\s+|\n+")


def clauses(text):
    norm = normalize(unwrap(text))
    return [c.strip() for c in _SENT_SPLIT_RE.split(norm) if c and c.strip()]


# Demo rapida sobre el ejemplo con vinetas visto en la Fase 2.
demo = ("Findings:\n"
        "> The contour of ACL is in a straight line but increase signal intensity, likely due to sprain\n"
        "> No definite tear of medial or lateral meniscus in this study")
for c in clauses(demo):
    print("-", c)

## C. Lexicon por hallazgo (ingles + espanol) y cues de negacion

Fuente de la metodologia (ya citada en RESOURCES.md): Ratner et al. 2016/Snorkel para el enfoque de weak supervision con funciones de etiquetado; los dos notebooks de referencia para el patron "assertion/negation" y el pooling multilenguaje (probar todos los idiomas a la vez, sin enrutar por idioma detectado).

Ambito de esta primera version: **solo ingles y espanol**, las dos lenguas dominantes segun el vocabulario medido en `02_eda_reports.ipynb` seccion F. El griego/cirilico (~12% de los informes por script, seccion D de esa misma fase) queda sin cobertura por ahora — cae en la abstencion (0.5) de `label_report()`, no en una respuesta incorrecta con confianza. Ampliar cobertura a mas idiomas es un candidato a iteracion futura, no bloqueante para este gate.

In [ ]:
_OA_PATHOLOGY_CUES = [
    r"osteoarthrit", r"osteoarthros", r"osteoartr", r"chondrosis",
    r"(cartilage|chondral) (loss|thinning|defect|fissur)",
    r"joint space narrowing", r"osteophyte", r"osteofito", r"spurring",
    r"pinzamiento", r"degenerat", r"adelgazamiento del cartilago",
    r"chondromalacia", r"condromalacia", r"subchondral cystic",
]

FINDING_LEXICON = {
    "acl_injury": dict(
        anatomy=[r"\bacl\b", r"anterior cruciate ligament", r"ligamento cruzado anterior", r"\blca\b"],
        pathology=[r"\btear", r"\btorn\b", r"ruptur", r"sprain", r"rotur", r"desgarr", r"esguinc", r"discontinuit"],
    ),
    "mcl_injury": dict(
        anatomy=[r"\bmcl\b", r"medial collateral ligament", r"ligamento colateral medial", r"ligamento lateral interno"],
        pathology=[r"\btear", r"\btorn\b", r"ruptur", r"sprain", r"rotur", r"desgarr", r"esguinc"],
    ),
    "medial_meniscus_tear": dict(
        anatomy=[r"medial meniscus", r"menisco medial"],
        pathology=[r"\btear", r"\btorn\b", r"rotur", r"desgarr", r"extrusion", r"extrusi[o0]n", r"macerat"],
    ),
    "lateral_meniscus_tear": dict(
        anatomy=[r"lateral meniscus", r"menisco lateral"],
        pathology=[r"\btear", r"\btorn\b", r"rotur", r"desgarr", r"extrusion", r"extrusi[o0]n", r"macerat"],
    ),
    "oa_medial_compartment": dict(
        anatomy=[r"medial compartment", r"medial femorotibial", r"medial femoral condyle",
                 r"medial tibial plateau", r"compartimento medial"],
        pathology=_OA_PATHOLOGY_CUES,
    ),
    "oa_lateral_compartment": dict(
        anatomy=[r"lateral compartment", r"lateral femorotibial", r"lateral femoral condyle",
                 r"lateral tibial plateau", r"compartimento lateral"],
        pathology=_OA_PATHOLOGY_CUES,
    ),
    "oa_patellofemoral_compartment": dict(
        anatomy=[r"patellofemoral", r"femoropatelar", r"femororrotulian", r"patelofemoral"],
        pathology=_OA_PATHOLOGY_CUES,
    ),
    "effusion": dict(
        anatomy=[r"\beffusion\b", r"derrame articular", r"\bderrame\b", r"efusi[o0]n"],
        pathology=[r"\beffusion\b", r"derrame", r"efusi[o0]n", r"fluid collection", r"joint fluid"],
    ),
    "synovitis": dict(
        anatomy=[r"synovit", r"sinovit", r"synovial (thickening|hypertrophy|proliferation)", r"engrosamiento sinovial"],
        pathology=[r"synovit", r"sinovit", r"synovial (thickening|hypertrophy|proliferation)", r"engrosamiento sinovial"],
    ),
    "bakers_cyst": dict(
        anatomy=[r"baker'?s? cyst", r"popliteal cyst", r"quiste de baker", r"quiste poplite"],
        pathology=[r"baker'?s? cyst", r"popliteal cyst", r"quiste de baker", r"quiste poplite"],
    ),
    "bone_contusion": dict(
        anatomy=[r"bone (marrow )?contusion", r"bone (marrow )?edema", r"contusi[o0]n [o0]sea", r"edema [o0]seo", r"edema medular"],
        pathology=[r"bone (marrow )?contusion", r"bone (marrow )?edema", r"contusi[o0]n [o0]sea", r"edema [o0]seo", r"edema medular"],
    ),
    "fracture": dict(
        anatomy=[r"\bfractur"],
        pathology=[r"\bfractur", r"cortical (break|disruption)", r"trabecular fracture"],
    ),
}

_NEGATION_CUES = [
    r"\bno\b", r"\bnot\b", r"\bwithout\b", r"\babsent\b", r"\bnormal\b",
    r"\bintact\b", r"\bunremarkable\b", r"within normal limit",
    r"no evidence of", r"no sign", r"negative for",
    r"\bsin\b", r"\bausen", r"\bninguna\b", r"\bnegativ",
    r"dentro de (los )?l[i0]mites normales", r"no se (observa|evidencia|aprecia)",
]
_NEGATION_RE = re.compile("|".join(_NEGATION_CUES))

_COMPILED_LEXICON = {
    finding: (re.compile("|".join(cues["anatomy"])), re.compile("|".join(cues["pathology"])))
    for finding, cues in FINDING_LEXICON.items()
}

## D. Primer intento de `label_report()` — y un bug real

Regla ingenua: dentro de una clausula que menciona la anatomia del hallazgo, si hay un cue de patologia, vota positivo — incluso si tambien hay un cue de negacion (razonamiento inicial: un termino especifico como "tear" pesa mas que una negacion generica). Validar esto contra el gold antes de asumir que es correcto.

In [ ]:
def label_report_v1(report_text, finding):
    anatomy_re, pathology_re = _COMPILED_LEXICON[finding]
    votes = []
    for c in clauses(report_text):
        if not anatomy_re.search(c):
            continue
        negated = bool(_NEGATION_RE.search(c))
        pathological = bool(pathology_re.search(c))
        if negated and not pathological:
            votes.append(0.0)
        elif pathological:  # gana la patologia incluso si tambien hay negacion
            votes.append(1.0)
    if not votes:
        return 0.5
    return 1.0 if max(votes) == 1.0 else 0.0


def evaluate_labeler(label_fn):
    y_true = gold[LABEL_COLS].reset_index(drop=True)
    y_true.columns = FINDINGS
    y_pred = pd.DataFrame({
        f: gold["Report"].apply(lambda t: label_fn(t, f)).reset_index(drop=True)
        for f in FINDINGS
    })
    return y_true, y_pred


y_true, y_pred_v1 = evaluate_labeler(label_report_v1)
print("Macro ROC-AUC (v1, naive):", macro_roc_auc(y_true, y_pred_v1))
print()
print(per_finding_roc_auc(y_true, y_pred_v1).sort_values())

**Bug real encontrado:** `effusion` sale con AUC **0.438 — peor que azar**. Causa: para hallazgos donde la entidad y la patologia son el mismo termino (`effusion`, `synovitis`, `baker's cyst`, `bone_contusion`), la clausula "no effusion" contiene la palabra "effusion", asi que dispara tanto el cue de negacion COMO el de patologia — y la regla v1 hace ganar a la patologia, votando positivo toda mencion negada. Exactamente el tipo de error que este proyecto existe para cazar antes de que llegue a `src/` (mismo espiritu que el bug de `ImagePositionPatient[2]` en Fase 1).

## E. Fix — la negacion gana siempre sobre la patologia ambigua

In [ ]:
def label_report_v2(report_text, finding):
    anatomy_re, pathology_re = _COMPILED_LEXICON[finding]
    votes = []
    for c in clauses(report_text):
        if not anatomy_re.search(c):
            continue
        if _NEGATION_RE.search(c):
            votes.append(0.0)
        elif pathology_re.search(c):
            votes.append(1.0)
    if not votes:
        return 0.5
    return 1.0 if max(votes) == 1.0 else 0.0


y_true, y_pred_v2 = evaluate_labeler(label_report_v2)
print("Macro ROC-AUC (v2, negacion prioritaria):", macro_roc_auc(y_true, y_pred_v2))
print()
auc_v2 = per_finding_roc_auc(y_true, y_pred_v2).sort_values()
print(auc_v2)
print()
print("Silence rate (fraccion de gold sin ninguna clausula que dispare el anatomy cue):")
print((y_pred_v2 == 0.5).mean().sort_values())

Mejora real: 0.629 -> 0.677, y ningun hallazgo por debajo de 0.5 ya. Quedan como mas debiles los tres compartimentos de OA (silence rate 70-95% — el lexicon de anatomia casi nunca dispara). Antes de asumir que el lexicon de OA esta mal, revisar informes reales positivos para ver que frases usan de verdad (no adivinar).

## F. Iterar OA con evidencia real, no con suposiciones

In [ ]:
pos_medial_oa = gold.loc[gold["Medial OA"] == 1.0, "Report"]
print(f"{len(pos_medial_oa)} filas positivas para Medial OA - inspeccionando la primera:\n")
print(pos_medial_oa.iloc[0][:1200])

Los informes en ingles SI usan encabezados de seccion tipo `MEDIAL COMPARTMENT:` seguidos de una linea `Medial compartment cartilage: ...`, pero la patologia real aparece con sinonimos que el lexicon v2 no cubria: "cartilage fissuring" (no solo "chondral fissuring"), "spurring" (no solo "osteophyte"), "subchondral cystic change", "chondrosis". Ampliar `_OA_PATHOLOGY_CUES` y la anatomia (agregar "medial femoral condyle"/"medial tibial plateau" como formas alternativas de nombrar el compartimento) con estos terminos vistos en los datos reales.

In [ ]:
# _OA_PATHOLOGY_CUES y la anatomia de oa_medial_compartment/oa_lateral_compartment
# en la celda C ya incluyen estos terminos (chondrosis, cartilage/chondral
# fissur, spurring, subchondral cystic, medial/lateral femoral condyle,
# medial/lateral tibial plateau) - fueron anadidos ahi tras esta inspeccion,
# para que el lexicon de la celda C sea directamente el que se valida abajo
# y el que se graduaria a src/labelers.py (sin una version v3 duplicada).

y_true, y_pred_final = evaluate_labeler(label_report_v2)  # mismo v2, lexicon ya ampliado en C
print("Macro ROC-AUC (lexicon final):", macro_roc_auc(y_true, y_pred_final))
print()
auc_final = per_finding_roc_auc(y_true, y_pred_final).sort_values()
print(auc_final)
print()
print("Silence rate final:")
print((y_pred_final == 0.5).mean().sort_values())

## G. Gate — ¿gana contra el baseline?

Baseline: prediccion constante 0.5 (equivalente a "no predecir nada", AUC=0.5 por construccion) para cada hallazgo. `macro_roc_auc` exige que ambas clases esten presentes — cierto en las 12 columnas del gold (minimo 9 positivos, MCL).

In [ ]:
baseline = pd.DataFrame({f: [0.5] * len(y_true) for f in FINDINGS})
print("Baseline (constante 0.5) macro AUC:", macro_roc_auc(y_true, baseline))
print("Labeler (lexicon final) macro AUC:  ", macro_roc_auc(y_true, y_pred_final))
print()
print("Por hallazgo, labeler vs. 0.5:")
print(auc_final)
print()
n_below_baseline = (auc_final < 0.5).sum()
print(f"Hallazgos por debajo del baseline: {n_below_baseline} / 12")

## Resumen y limitaciones — pendiente de revision antes de tocar `src/labelers.py`

**Resultado:** macro ROC-AUC 0.688 vs. 0.5 del baseline constante; los 12 hallazgos individualmente por encima de 0.5 (el mas debil: `oa_lateral_compartment` en 0.553, arrastrado por un silence rate del 90%).

**Limitaciones honestas, no resueltas en esta pasada:**
- **n=58 es un set de validacion pequeno** (tan solo 9 positivos en MCL). Los numeros de AUC por hallazgo son estimaciones ruidosas — seguir ajustando el lexicon contra este mismo set repetidamente arriesga sobreajustarlo a estos 58 casos concretos en vez de generalizar.
- **Solo ingles y espanol tienen cobertura de cues.** Griego y cirilico (~12% de los 4,407 informes por script, Fase 2 seccion D) caen siempre en la abstencion (0.5) — no se sabe si el labeler acierta o falla ahi, solo que no vota.
- **Negacion por co-ocurrencia en la clausula, no direccional.** Un clausula larga con una negacion de un hallazgo y una afirmacion de otro en la misma frase podria confundir el scope — los notebooks de referencia lo resuelven con "directional negation"; no implementado aqui.
- **`oa_lateral_compartment` sigue siendo el mas debil** (silence rate ~90%) pese a ampliar el lexicon en la seccion F.

**Decision:** el labeler gana el gate cuantitativo (mejora en las 12 columnas, no solo en macro). Antes de escribir esto en `src/labelers.py::label_report()`/`label_reports()` con sus tests correspondientes, pendiente de que el usuario revise este notebook y confirme si procede graduarlo tal cual o iterar mas primero.